# Look at results from metab x antibiotic use regressions on clinical chemistries

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator

import arivale_data_interface as adi
# apply settings so the lib knows where the data is
adi.apply_isb_config()

import re
import ast
from rich.progress import track
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.robust.norms as norms

import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# Now you can import your file
from regression_functions import *

## Read in raw data for plotting

In [95]:
# ---- Read + merge ------------
labs_clean = pd.read_csv("../data/clincal_chems_abx_users_clean.csv", dtype={'public_client_id': str})
metabolites_merged = pd.read_csv("../data/metabolites_merged_used_for_regressions.csv", dtype={'public_client_id': str})

# Grab vendor
quest = labs_clean[labs_clean['vendor'] == 'Quest']
labcorp = labs_clean[labs_clean['vendor'] == 'LCA']

# Prepare labs for merge
lab_features = labs_clean.columns[31:].tolist()
cols = lab_features + ['vendor']
labs_clean.set_index('public_client_id', inplace=True)
labs_for_merge = labs_clean[cols].reset_index()

# Merge and set year var
labs_mets = metabolites_merged.merge(labs_for_merge, on='public_client_id', how='left')
labs_mets['abx_year'] = labs_mets['abx_recency'].apply(lambda x: "none" if x == "none" else "past_year")

metabolite_features = metabolites_merged.columns[27:].tolist()
not_offered_quest = quest[lab_features].isna().all().index[quest[lab_features].isna().all()].tolist()
not_offered_labcorp = labcorp[lab_features].isna().all().index[labcorp[lab_features].isna().all()].tolist()
individual_vendors = not_offered_labcorp + not_offered_quest
shared_labs = list(set(lab_features) - set(individual_vendors))

## Read in results from interactions

In [96]:
interaction_results = pd.read_csv("../data/metabolite_abx_interactions_on_labs.csv")

In [97]:
sig_interactions = interaction_results[
(interaction_results['q_value'] < 0.1) & (interaction_results['q_value_global'] < 0.1)
].sort_values(by=['q_value_global','q_value'])

In [98]:
# sig_interactions

### Merge annotations

In [99]:
anns_snapshot = pd.read_csv("../data/metabolites_snapshot_updated.csv")
anns_snapshot = anns_snapshot.rename(columns={'feature':'metabolite'})

In [100]:
# Add annotations to the significant hits
sig_ints_anns = sig_interactions.merge(anns_snapshot, on='metabolite', how='left')
# sig_ints_anns.chemistry.value_counts()

In [101]:
lmplot_chems = sig_ints_anns.chemistry.value_counts()[sig_ints_anns.chemistry.value_counts() <= 3].index.tolist()
barbell_chems = sig_ints_anns.chemistry.value_counts()[sig_ints_anns.chemistry.value_counts() > 3].index.tolist()

lmplot_chems_df = sig_ints_anns[sig_ints_anns.chemistry.isin(lmplot_chems)]
barbell_chems_df = sig_ints_anns[sig_ints_anns.chemistry.isin(barbell_chems)]

lmplot_tuples = list(zip(lmplot_chems_df.chemistry, lmplot_chems_df.metabolite))
len(lmplot_tuples)

33

In [102]:
barbell_chems_df.chemistry.unique()

array(['TriglycerideHDLRatio', 'TRIGLYCERIDES', 'IL_6',
       'TOTALNEUTROPHILS', 'LYMPHOCYTES'], dtype=object)

In [103]:
covariates = ["sex", "age", "BMI_CALC", "meds_cholesterol",
              "meds_blood_sugar", "PC1", "PC2", "PC3", "PC4", "PC5",
              "centered_eGFR", "vendor"]

## Check if association relies on high leverage points

In [104]:
# metabolites = adi.get_snapshot("metabolomics_corrected").sort_values(by="days_in_program")

In [105]:
def interaction_influence(df, lab, metabolite, covars,
                          abx="abx_year"):
    """Refit one lab~abx*metabolite+covars model, return influence on the
    interaction term + a leverage-dropped refit for comparison."""
    d = df[[lab, abx, metabolite] + covars].dropna().copy()
    n = len(d)

    rhs = f"C({abx}) * {metabolite}" + ("" if not covars else " + " + " + ".join(covars))
    fit = smf.ols(f"Q('{lab}') ~ {rhs}", data=d).fit()

    # find the interaction column in the design (patsy names it with ':')
    inter = [c for c in fit.params.index if ":" in c and metabolite in c]
    assert len(inter) == 1, f"interaction term not uniquely found: {inter}"
    inter = inter[0]

    infl = fit.get_influence()
    cooks = infl.cooks_distance[0]                       # (n,)
    dfb   = infl.dfbetas[:, list(fit.params.index).index(inter)]  # (n,) for the interaction

    flag = (np.abs(dfb) > 2/np.sqrt(n)) | (cooks > 4/n)

    # refit dropping the flagged high-influence points
    fit_drop = smf.ols(f"Q('{lab}') ~ {rhs}", data=d.loc[~flag]).fit()

    return {
        "lab": lab, "metabolite": metabolite, "n": n, "n_dropped": int(flag.sum()),
        "beta_full":  fit.params[inter],      "p_full":  fit.pvalues[inter],
        "beta_drop":  fit_drop.params[inter], "p_drop":  fit_drop.pvalues[inter],
        "max_abs_dfbetas": np.abs(dfb).max(),
        "max_cooks": cooks.max(),
        # the actual influential rows, for eyeballing
        "influential_idx": d.index[flag].tolist(),
    }

In [106]:
# # run across the 117 hits
# lev_results = pd.DataFrame(
#     interaction_influence(labs_mets, r.chemistry, r.metabolite, covariates)
#     for r in sig_ints_anns.itertuples() 
# )

In [107]:
# lev_results.to_csv('../data/abx_metab_lab_interactions_leverage.csv', index=False)

In [108]:
lev_results = pd.read_csv('../data/abx_metab_lab_interactions_leverage.csv')

In [109]:
# the honest summary you report:
lev_results["sign_flip"]   = np.sign(lev_results.beta_full) != np.sign(lev_results.beta_drop)
lev_results["lost_sig"]    = (lev_results.p_full < 0.05) & (lev_results.p_drop >= 0.05)
lev_results["leverage_dependent"] = lev_results.sign_flip | lev_results.lost_sig
print(lev_results.leverage_dependent.value_counts())

leverage_dependent
False    76
True     39
Name: count, dtype: int64


In [110]:
lev_results['influential_idx'] = lev_results['influential_idx'].apply(ast.literal_eval)

In [111]:
lev_results[lev_results.leverage_dependent == True].lab.value_counts()

lab
IL_6                    14
TRIGLYCERIDES           12
TriglycerideHDLRatio     5
GLYCOHEMOGLOBINA1C       1
HOMA_IR                  1
QUICKI                   1
ARSENIC_BLOOD            1
ALAT_SGPT_               1
CRPHIGHSENSITIVITY       1
CREATININEENZ_SER        1
FERRITIN                 1
Name: count, dtype: int64

In [112]:
# lev_results[lev_results.leverage_dependent == False].lab.value_counts()

## Check whether associations survive using robust regression (curbing outlier influence)

In [113]:
def robust_regression(df, lab, metabolite, covars, abx="abx_year"):
    """Refit model using robust regression"""
    d = df[[lab, abx, metabolite] + covars].dropna().copy()
    n = len(d)

    rhs = f"C({abx}) * {metabolite}" + ("" if not covars else " + " + " + ".join(covars))
    fit_rlm = smf.rlm(f"Q('{lab}') ~ {rhs}", data=d, M=norms.HuberT()).fit()

    inter = [c for c in fit_rlm.params.index if ":" in c and metabolite in c]
    assert len(inter) == 1, f"interaction term not uniquely found: {inter}"
    inter = inter[0]
    
    return {
        "lab": lab,
        "metabolite": metabolite,
        "n": n,
        "beta_robust": fit_rlm.params[inter],
        "p_robust": fit_rlm.pvalues[inter]
    }

In [114]:
# run across the 117 hits
results_robust = pd.DataFrame(
    robust_regression(labs_mets, r.chemistry, r.metabolite, covariates)
    for r in sig_ints_anns.itertuples() 
)

results_robust.to_csv("../data/abx_metab_interaction_robust.csv", index=False)
#results_robust = pd.read_csv("../data/abx_metab_interaction_robust.csv")

## Merge and check interactions that pass both tests

In [115]:
lev_results.set_index(['lab','metabolite','n'], inplace=True)
results_robust.set_index(['lab','metabolite','n'], inplace=True)

In [116]:
hits = pd.merge(lev_results, results_robust, left_index=True, right_index=True, how='inner')

In [117]:
hits.reset_index(inplace=True)

In [118]:
# Passes leverage
hits["leverage_robust"] = ~(hits.sign_flip | hits.lost_sig)

# Helper columns/passes robust
hits["sign_flip_rob"] = np.sign(hits.beta_full) != np.sign(hits.beta_robust)
hits["lost_sig_rob"]  = hits.p_robust >= 0.05          # nominal, NOT re-FDR'd
hits["huber_robust"]  = ~(hits.sign_flip_rob | hits.lost_sig_rob)

# pass BOTH ---
hits["passes_both"] = hits.leverage_robust & hits.huber_robust

print(hits.passes_both.value_counts())
tier1 = hits[hits.passes_both]    

passes_both
False    76
True     39
Name: count, dtype: int64


In [119]:
# tier1.lab.value_counts()

In [120]:
filtered_hits = list(zip(tier1.lab, tier1.metabolite))

In [121]:
sig_ints_anns.reset_index(inplace=True)

In [122]:
filtered_hits_df = sig_ints_anns[sig_ints_anns.set_index(['chemistry','metabolite']).index.isin(filtered_hits)]

## Next, let's check for a monotonic trend within abx recency groups within our hits

In [123]:
# ordinal recency score: increasing = more recent antibiotic exposure
recency_score = {"none": 0, "twelve": 1, "three": 2}

In [124]:
def recency_trend_test(chem, metabolite, data, covars,
                       recency_col="abx_recency", score_map=recency_score,
                       binary_beta=None):
    """Refit one hit under the 3-level recency model (ordinal + categorical).

    Parameters
    ----------
    chem, metabolite : str      the hit's outcome + predictor columns
    data : pd.DataFrame          merged per-person frame (must contain recency_col)
    covars : list[str]           RHS covariate terms, e.g.
                                 ["C(sex)","age","BMI_CALC","C(meds_cholesterol)",
                                  "C(meds_blood_sugar)","PC1","PC2","PC3","PC4","PC5",
                                  "centered_eGFR","C(vendor)"]
    binary_beta : float|None     the hit's slope-difference sign from the binary
                                 model; if given, we check the trend agrees in sign.

    Returns
    -------
    dict with the ordinal-trend estimate, the two categorical interaction coefs,
    per-level Ns, and boolean concordance flags (is_monotonic, sign_concordant).
    """
    keep = [chem, metabolite, recency_col] + _cols(covars)
    d = data[keep].dropna().copy()
    # Make sure our levels are there
    d = d[d[recency_col].isin(score_map)]                 
    d["abx_year"] = d[recency_col].map(score_map).astype(float)
    # Find n by recencey group to check for super small sample sizes
    n_by_level = d[recency_col].value_counts().to_dict()

    covar_rhs = (" + " + " + ".join(covars)) if covars else ""

    # First person the original regression
    fit_ord = smf.ols(
        f"Q('{chem}') ~ abx_year * Q('{metabolite}') {covar_rhs}", data=d
    ).fit()
    ord_term = _find_interaction(fit_ord, "abx_year", metabolite)
    trend_beta = fit_ord.params[ord_term]
    trend_p = fit_ord.pvalues[ord_term]

    # Next check the coefficients/slope when fitting the 3-level abx recency
    fit_cat = smf.ols(
        f"Q('{chem}') ~ C({recency_col}, Treatment(reference='none')) "
        f"* Q('{metabolite}') {covar_rhs}", data=d
    ).fit()
    b12 = _cat_interaction_coef(fit_cat, "twelve", metabolite)   # 12mo vs none
    b3 = _cat_interaction_coef(fit_cat, "three", metabolite)     # 3mo vs none

    # monotonic if 12mo strictly between none and 3mo, in either direction
    is_monotonic = (b12 is not None and b3 is not None and
                    ((0 <= b12 <= b3) or (b3 <= b12 <= 0)))

    sign_concordant = (binary_beta is None) or (np.sign(trend_beta) == np.sign(binary_beta))

    return {
        "chemistry": chem, "metabolite": metabolite, "n": int(d.shape[0]),
        "n_none": n_by_level.get("none", 0),
        "n_twelve": n_by_level.get("twelve", 0),
        "n_three": n_by_level.get("three", 0),
        "trend_beta": trend_beta, "trend_p": trend_p,   # per-hit concordance
        "beta_12mo_vs_none": b12, "beta_3mo_vs_none": b3,
        "is_monotonic": bool(is_monotonic),             # 12mo lands between none and 3mo
        "sign_concordant": bool(sign_concordant),       # trend agrees with binary hit
    }

In [125]:
def run_recency_trend(hits, data, covars, beta_col="beta"):
    """Loop recency_trend_test over the hits table; returns an annotated DataFrame.

    hits : the 117-row table (needs 'chemistry','metabolite', optionally beta_col)
    """
    out = []
    for _, h in hits.iterrows():
        bb = h[beta_col] if beta_col in hits.columns else None
        try:
            out.append(recency_trend_test(h["chemistry"], h["metabolite"],
                                          data, covars, binary_beta=bb))
        except Exception as e:                          # keep going; log the failure
            out.append({"chemistry": h["chemistry"], "metabolite": h["metabolite"],
                        "error": str(e)})
    res = pd.DataFrame(out)
    # the clean "dose-response" tag: monotonic AND sign-concordant with the binary hit
    if {"is_monotonic", "sign_concordant"}.issubset(res.columns):
        res["clean_gradient"] = res["is_monotonic"] & res["sign_concordant"]
    return res

In [126]:
def _find_interaction(fit, left, metabolite):
    """Interaction term containing both `left` (e.g. abx_year) and the metabolite."""
    hits = [c for c in fit.params.index
            if ":" in c and left in c and metabolite in c]
    assert len(hits) == 1, f"interaction term not unique: {hits}"
    return hits[0]

def _cat_interaction_coef(fit, level, metabolite):
    """Coef for the (recency==level : metabolite) interaction, or None if absent."""
    for c in fit.params.index:
        if ":" in c and metabolite in c and f"[T.{level}]" in c:
            return fit.params[c]
    return None

def _cols(covariates):
    """Strip C(...) wrappers to raw column names for dropna."""
    import re
    out = []
    for term in covariates:
        m = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", term)
        out.append(m[0] if m else term)
    return list(dict.fromkeys(out))

In [127]:
recency_check = run_recency_trend(sig_interactions, labs_mets, covariates)

In [128]:
recency_check = recency_check.drop(columns='n')

In [129]:
recency_check.rename(columns={'chemistry':'lab'}, inplace=True)

In [130]:
recency_check.set_index(['lab', 'metabolite'], inplace=True)
hits.set_index(['lab', 'metabolite'], inplace=True)

In [131]:
hits_recency_check = hits.join(recency_check, how='inner')

In [132]:
hits_recency_check.to_csv("../data/abx_metab_interaction_all_hits_with_test_columns.csv")
hits_recency_check = pd.read_csv("../data/abx_metab_interaction_all_hits_with_test_columns.csv")

In [133]:
monotonic_hits = hits_recency_check[(hits_recency_check.passes_both) & (hits_recency_check.clean_gradient)].reset_index()

In [134]:
monotonic_tuples = list(zip(monotonic_hits.lab, monotonic_hits.metabolite))

In [135]:
filtered_hits_df = sig_ints_anns[sig_ints_anns.set_index(['chemistry','metabolite']).index.isin(monotonic_tuples)].drop(
    columns='index'
)

In [136]:
len(filtered_hits_df.sort_values(by=['chemistry','CHEMICAL_NAME']))

20

In [137]:
fig, bar = make_recency_barbell(labs_mets, filtered_hits_df, covariates)

In [138]:
#filtered_hits_df.to_csv("../data/abx_metab_interaction_hits_filtered.csv", index=False)

In [139]:
#filtered_hits_df = pd.read_csv("../data/abx_metab_interaction_hits_filtered.csv")

In [160]:
# test_lev = lev_results.reset_index()

In [161]:
# test_lev

In [159]:
look_at = ['LYMPHOCYTES','TOTALNEUTROPHILS','TOTALNEUTROPHILSAB','LYMPHOCYTESABSOLUTE']
mets = ['metabolite_796', 'metabolite_100015593', 'metabolite_100000936','metabolite_999946905']
look_at_df = interaction_results[interaction_results.chemistry.isin(look_at)].copy()
look_at_mets = look_at_df[look_at_df.metabolite.isin(mets)]
look_at_mets

,metabolite,beta,t_stat,p_value,n,q_value,chemistry,q_value_global
38847,metabolite_796,-1.622079,-3.976742,0.000073,1493,0.023659,LYMPHOCYTES,0.067391
38983,metabolite_100000936,-1.584681,-3.880102,0.000109,1508,0.026396,LYMPHOCYTES,0.087101
39452,metabolite_100015593,-1.716808,-4.113234,0.000041,1392,0.020023,LYMPHOCYTES,0.047992
39654,metabolite_999946905,-1.376400,-3.371293,0.000767,1507,0.123918,LYMPHOCYTES,0.249606
49506,metabolite_796,2.026820,4.510924,0.000007,1493,0.005689,TOTALNEUTROPHILS,0.011971
49642,metabolite_100000936,1.780791,3.960653,0.000078,1508,0.015225,TOTALNEUTROPHILS,0.070586
50111,metabolite_100015593,1.944708,4.221997,0.000026,1392,0.008335,TOTALNEUTROPHILS,0.032196
50313,metabolite_999946905,1.760594,3.917674,0.000093,1507,0.015225,TOTALNEUTROPHILS,0.080253
68886,metabolite_796,0.179154,3.024436,0.002534,1493,0.343512,TOTALNEUTROPHILSAB,0.394912
69022,metabolite_100000936,0.175956,2.953558,0.003191,1508,0.343512,TOTALNEUTROPHILSAB,0.419958


In [149]:
filtered_hits_df[filtered_hits_df.chemistry.isin(['TOTALNEUTROPHILS','LYMPHOCYTES'])].metabolite.value_counts().index

Index(['metabolite_796', 'metabolite_100015593', 'metabolite_100000936',
       'metabolite_999946905'],
      dtype='object', name='metabolite')

In [141]:
labs_mets.columns # metabolite_100000936, metabolite_999946905

Index(['public_client_id', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5',
       'days_in_program', 'meds_blood_sugar', 'meds_cholesterol',
       'abx_recency',
       ...
       'URICACID', 'VITAMIND_25_OHTOT', 'VITAMIND3_25_OH', 'WHITECELLCOUNT',
       'ZINC', 'ZINC_RBC', 'QUICKI', 'EPA_AA', 'vendor', 'abx_year'],
      dtype='object', length=1092)

In [142]:
cbc = labs_mets[['LYMPHOCYTES','TOTALNEUTROPHILS','WHITECELLCOUNT','TOTALNEUTROPHILSAB','LYMPHOCYTESABSOLUTE', 'metabolite_100000936']].copy()

In [143]:
round(cbc['LYMPHOCYTES']/100 * cbc['WHITECELLCOUNT'], 2) # Confirming that LYMPHOCYTES and TOTALNEUTROPHILS cols are CBC %

0       1.50
1       1.64
2       1.24
3       1.36
4       1.73
        ... 
1594    1.01
1595    1.88
1596    1.47
1597    0.96
1598    2.16
Length: 1599, dtype: float64

In [91]:
formula_lymphs_abs = ("LYMPHOCYTESABSOLUTE ~ C(sex) + age + BMI_CALC + C(abx_year)*metabolite_100000936 +"
                      "C(meds_cholesterol) + C(meds_blood_sugar) + PC1 + PC2 + PC3 + PC4 + PC5+ "
                      "centered_eGFR + C(vendor)")
formula_neut_abs = ("TOTALNEUTROPHILSAB ~ C(sex) + age + BMI_CALC + C(abx_year)*metabolite_100000936 +"
                      "C(meds_cholesterol) + C(meds_blood_sugar) + PC1 + PC2 + PC3 + PC4 + PC5+ "
                      "centered_eGFR + C(vendor)")

In [93]:
model_lymphsabs = smf.ols(formula_lymphs_abs, data=labs_mets).fit()
model_lymphsabs.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     LYMPHOCYTESABSOLUTE   R-squared:                       0.108
Model:                             OLS   Adj. R-squared:                  0.099
Method:                  Least Squares   F-statistic:                     12.07
Date:                 Mon, 07 Sep 2026   Prob (F-statistic):           1.18e-28
Time:                         17:38:41   Log-Likelihood:                -1151.7
No. Observations:                 1508   AIC:                             2335.
Df Residuals:                     1492   BIC:                             2420.
Df Model:                           15                                         
Covariance Type:             nonrobust                                         
=================================================================================================================
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
Intercept                                         1.6341      0.095     17.242      0.000       1.448       1.820
C(sex)[T.M]                                      -0.0815      0.029     -2.769      0.006      -0.139      -0.024
C(abx_year)[T.past_year]                          0.0007      0.028      0.026      0.979      -0.055       0.057
C(meds_cholesterol)[T.Yes]                        0.0138      0.043      0.317      0.751      -0.071       0.099
C(meds_blood_sugar)[T.Yes]                        0.1713      0.070      2.446      0.015       0.034       0.309
C(vendor)[T.Quest]                               -0.0150      0.031     -0.479      0.632      -0.077       0.047
age                                              -0.0059      0.001     -3.985      0.000      -0.009      -0.003
BMI_CALC                                          0.0166      0.002      7.299      0.000       0.012       0.021
metabolite_100000936                              0.0578      0.018      3.271      0.001       0.023       0.092
C(abx_year)[T.past_year]:metabolite_100000936    -0.0402      0.028     -1.417      0.157      -0.096       0.015
PC1                                               0.0212      0.107      0.198      0.843      -0.189       0.232
PC2                                               0.2592      0.087      2.988      0.003       0.089       0.429
PC3                                               0.3775      0.106      3.575      0.000       0.170       0.585
PC4                                               0.0476      0.102      0.468      0.640      -0.152       0.247
PC5                                              -0.1761      0.086     -2.052      0.040      -0.344      -0.008
centered_eGFR                                     0.0017      0.001      1.512      0.131      -0.001       0.004
==============================================================================
Omnibus:                      583.277   Durbin-Watson:                   2.079
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             5050.536
Skew:                           1.563   Prob(JB):                         0.00
Kurtosis:                      11.403   Cond. No.                         481.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [92]:
model_neut_abs = smf.ols(formula_neut_abs, data=labs_mets).fit()
model_neut_abs.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     TOTALNEUTROPHILSAB   R-squared:                       0.173
Model:                            OLS   Adj. R-squared:                  0.165
Method:                 Least Squares   F-statistic:                     20.86
Date:                Mon, 07 Sep 2026   Prob (F-statistic):           6.53e-52
Time:                        17:37:06   Log-Likelihood:                -2270.3
No. Observations:                1508   AIC:                             4573.
Df Residuals:                    1492   BIC:                             4658.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
=================================================================================================================
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
Intercept                                         1.4371      0.199      7.222      0.000       1.047       1.827
C(sex)[T.M]                                      -0.2454      0.062     -3.971      0.000      -0.367      -0.124
C(abx_year)[T.past_year]                         -0.0329      0.060     -0.550      0.582      -0.150       0.084
C(meds_cholesterol)[T.Yes]                        0.1033      0.091      1.132      0.258      -0.076       0.282
C(meds_blood_sugar)[T.Yes]                        0.3552      0.147      2.415      0.016       0.067       0.644
C(vendor)[T.Quest]                                0.4279      0.066      6.497      0.000       0.299       0.557
age                                              -0.0016      0.003     -0.520      0.603      -0.008       0.005
BMI_CALC                                          0.0680      0.005     14.254      0.000       0.059       0.077
metabolite_100000936                             -0.0295      0.037     -0.795      0.427      -0.102       0.043
C(abx_year)[T.past_year]:metabolite_100000936     0.1760      0.060      2.954      0.003       0.059       0.293
PC1                                               0.0814      0.225      0.361      0.718      -0.361       0.524
PC2                                              -0.5598      0.182     -3.072      0.002      -0.917      -0.202
PC3                                               0.5538      0.222      2.498      0.013       0.119       0.989
PC4                                               0.0795      0.214      0.372      0.710      -0.339       0.498
PC5                                              -0.2947      0.180     -1.635      0.102      -0.648       0.059
centered_eGFR                                     0.0049      0.002      2.083      0.037       0.000       0.010
==============================================================================
Omnibus:                      308.091   Durbin-Watson:                   1.942
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              705.983
Skew:                           1.128   Prob(JB):                    4.99e-154
Kurtosis:                       5.479   Cond. No.                         481.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""